# Model Arena · Slow System en ADA

Elige un perfil de corrida. Cada perfil prepara su universo, optimiza saldo y variación, entrena sus modelos ganadores y genera pronósticos MAD y salidas MAC 2. Los archivos en `private/` son específicos de ADA.

### Presupuesto por grupo

| Run | Modelos globales | Trials por objetivo |
| --- | --- | --- |
| 01 | NLinear, DLinear, RLinear | 8 por modelo en X0 |
| 02 | NBEATS, NHITS, TiDE | 8 por modelo. NBEATS usa X0. NHITS y TiDE reparten 2 por X0–X3 |
| 03 | Encoder–Decoder directo | 6 por X0–X3 |
| 04 | Encoder–Decoder residual sobre base fija | 6 por X0–X3 |
| 05 | Encoder–Decoder residual sobre base aprendida | 6 por X0–X3 |
| 06 | LightGBM global directo | 6 por X0–X3 |

Cada corrida tiene 24 trials de saldo y 24 de variación, con hasta 144 ajustes F1–F3. Los grupos 03–05 exploran MLP, GRU y TFT compacto como alternativas del encoder.

In [ ]:
from pathlib import Path
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

VERSION = "0.1.10"
# La instalación conserva los índices y credenciales que ADA ya tenga configurados.
# Si el índice corporativo aún no replica la versión, usa el wheel adjunto.
WHEEL = next(iter(Path.cwd().glob("dist/model_arena_sdk-0.1.10-py3-none-any.whl")), None)
try:
    installed = version("model-arena-sdk")
except PackageNotFoundError:
    installed = None
if installed != VERSION:
    requirement = str(WHEEL.resolve()) if WHEEL else f"model-arena-sdk=={VERSION}"
    command = [sys.executable, "-m", "pip", "install", requirement]
    if WHEEL:
        command.extend(["--find-links", str(WHEEL.parent.resolve())])
    subprocess.run(command, check=True)
from model_arena import __version__
assert __version__ == VERSION, "Reinicia el kernel: quedó importada una versión anterior"
print("Model Arena", __version__, "· Python", sys.version.split()[0])

## Crear o continuar un proyecto
Una corrida nueva genera el calendario con Calendar Studio y private/festivos.csv. Revisa y mantén ese insumo para el período que utilices; el generador no obtiene festivos oficiales. La vista MAC conserva 243 variables. Cada corrida congela datos, reglas, versión y hashes.
Al indicar PROJECT_PATH se reutiliza el calendario congelado de esa corrida, sin regenerarlo.

In [ ]:
from datetime import datetime, timezone
from model_arena import RunSpec
from model_arena.profiles.mac import MacProject
from uuid import uuid4
import torch

PROFILE = "01_lineales"  # Usa el nombre de un YAML en profiles/.
DEVICE = "cuda"  # Para 06_lightgbm selecciona "cpu".
PROJECT_PATH = ""  # Ruta de un proyecto creado aquí para continuar; vacío crea uno nuevo.
base = Path.cwd().resolve()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Seleccionaste CUDA pero el kernel no tiene PyTorch con GPU. Configura el runtime aprobado de ADA.")
if PROJECT_PATH:
    project = MacProject(PROJECT_PATH)
    project.verify()
else:
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_spec = RunSpec.from_yaml(base / "profiles" / (PROFILE + ".yaml"))
    from calendar_studio import CalendarStudio, load_holidays
    calendar = CalendarStudio.from_profile("mac_legacy_v1").generate(
        start="2022-01-01", end="2027-12-31",
        holidays=load_holidays(base / "private/festivos.csv", date_col="fecha"),
    ).view("mac_legacy_v1")
    calendar.validate().raise_for_errors()
    project = MacProject.create(
        base / "runs" / (f"mac_{run_spec.name}_{suffix}_{uuid4().hex[:8]}"),
        training_config=base / "private/training.json",
        publication_config=base / "private/publication.json",
        calendar=calendar,
        device=DEVICE,
        run_spec=run_spec,
    )
print("PROJECT_PATH =", repr(str(project.path)))
print(project.status())
plan = project.plan()
print("Plan HPO:", {k: v for k, v in plan.items() if k != "studies"})
info=project.calendar_provenance()
print("Calendario congelado:", {k:info.get(k) for k in ["generator","generator_version","profile","view","start","end","rows","feature_count","calendar_sha256"]})


## 2. Preparación del universo

Ingesta Engine 1 desde Athena o desde el parquet explícito de configuración. `requested_accounts=0` y `account_limit=0`: se usan todas las entrenables elegibles. Las atípicas se conservan para zero-shot. La segmentación por entropía mantiene la fórmula del perfil existente.

In [ ]:
project.prepare()
import json
print(json.loads((project.path / "run/receipts/ingesta.json").read_text())["status"])

### Presupuesto de la corrida

Revisa `trials_by_objective` y las entradas de cada estudio antes de lanzar el HPO. El YAML del grupo declara el reparto. `private/training.json` define los rangos comunes y las ventanas.

In [ ]:
project.tune()
import polars as pl
curvas = project.path / "run/curvas"
candidates = json.loads((curvas / "candidatos.json").read_text())
print("Candidatos completos:", len(candidates))
audit = pl.read_parquet(curvas / "hpo_trials.parquet")
print(audit.group_by("input", "state").len().sort("input", "state"))
print("Partes de loss por cuenta:", len(list((curvas / "loss_entities").rglob("*.parquet"))))


## 4. Ganador por cuenta y objetivo

Se asigna un ganador distinto para saldo y variación cuando la evidencia lo favorece. Las cuentas atípicas se asignan usando evidencia de su segmento y las reglas de fallback.

In [ ]:
project.select()
winners = pl.read_parquet(curvas / "winners_FUTURO.parquet")
print(winners.group_by("cohort").len())
print("Configuraciones únicas que requieren ajuste final:", winners["candidate"].n_unique())
winners.head()

## 5. Ajuste final

Cada configuración ganadora se ajusta con el universo entrenable y toda la historia disponible. El perfil fija 75 épocas para redes y hasta 75 rondas para LightGBM.

In [ ]:
project.fit()
print(project.status())

## 6. Pronóstico con modelos congelados

`predict` exige checkpoints existentes. No realiza entrenamiento. FUTURO tiene veinte fechas operativas y dos objetivos; carece de observados futuros para construir validación.

In [ ]:
forecast_path = project.predict()
print("Pronósticos:", forecast_path)
pl.read_parquet(forecast_path).head()

## 7. Postprocessing y layouts

Se generan las curvas históricas con checkpoints admisibles al corte del mismo candidato ganador. La evidencia sigue siendo de selección. Se construyen los alertamientos y las salidas MAC; las cuentas atípicas se evalúan como entrenables en alertamiento, manteniendo su procedencia zero-shot.

Esta etapa todavía no publica en AWS.

In [ ]:
release = project.postprocess()
manifest = json.loads((release / "publication.json").read_text())
print("Publication ID:", manifest["publication_id"])
for dataset in ["engine2", "engine3", "training_learning", "training_entities"]:
    item = manifest["datasets"].get(dataset, {})
    print(dataset, "filas:", item.get("rows"), "columnas:", len(item.get("columns", [])))

## 8. Publicación explícita en ADA

Activa `PUBLISH_TO_ADA` para publicar el run en el destino sandbox de `private/publication.json`. La publicación del SDK en PyPI y GitHub sigue su propio proceso.

In [ ]:
PUBLISH_TO_ADA = False
if PUBLISH_TO_ADA:
    receipt = project.publish()
    assert receipt["status"] == "PUBLISHED"
    assert receipt["publication_id"] == manifest["publication_id"]
    print("Publicación verificada:", receipt["publication_id"])

## 9. Abrir Observatory

El botón abre el monitor Athena existente. A partir de aquí las consultas no entrenan modelos. Para futuras sesiones usa el notebook Fast sin abrir el entorno de entrenamiento.

In [ ]:
if PUBLISH_TO_ADA:
    monitor = project.observe()
    print("PID del monitor:", monitor.pid)